In [10]:
import boto3
import os
from config.app import Config

config = Config()

In [11]:
s3_client = boto3.client(
    's3',
    region_name=config.awss3.region,
    aws_access_key_id=config.awss3.access_key_id,
    aws_secret_access_key=config.awss3.secret_access_key
)

In [ ]:
def explorar_bucket(prefix: str = ""):
    """
    Lista los objetos dentro del bucket configurado.
    :param prefix: Ruta o 'carpeta' dentro del bucket para filtrar.
    """
    try:
        response = s3_client.list_objects_v2(
            Bucket=config.awss3.bucket,
            Prefix=prefix
        )
        
        if 'Contents' in response:
            print(f"Objetos en '{config.awss3.bucket}' (Filtro: '{prefix}'):\n")
            for item in response['Contents']:
                # Muestra el nombre del archivo, tamaño en KB y fecha de modificación
                size_kb = round(item['Size'] / 1024, 2)
                print(f" - {item['Key']} ({size_kb} KB) | Última mod: {item['LastModified']}")
        else:
            print(f"El bucket '{config.awss3.bucket}' está vacío o no se encontraron objetos con el prefijo '{prefix}'.")

    except Exception as e:
        print(f"Error al explorar el bucket: {e}")

In [ ]:
explorar_bucket()

In [12]:
from collections import defaultdict

def resumir_estructura_s3(prefix: str = ""):
    """
    Recorre paginando todo el bucket y agrupa la cantidad de archivos 
    y tamaño total por cada tabla/dataset principal.
    """
    paginator = s3_client.get_paginator('list_objects_v2')
    page_iterator = paginator.paginate(
        Bucket=config.awss3.bucket,
        Prefix=prefix
    )

    # Diccionarios para acumular conteo y tamaño
    dataset_counts = defaultdict(int)
    dataset_sizes = defaultdict(float)
    total_files = 0
    total_size_bytes = 0

    for page in page_iterator:
        if 'Contents' in page:
            for item in page['Contents']:
                key = item['Key']
                size = item['Size']
                
                total_files += 1
                total_size_bytes += size

                # Identificar la ruta o dataset principal
                # Ej: 'data/call_center_interactions/year=2023/...' -> 'data/call_center_interactions/'
                parts = key.split('/')
                if len(parts) > 2:
                    dataset_key = f"{parts[0]}/{parts[1]}/"
                else:
                    dataset_key = key  # Archivo suelto en raíz/data, ej: 'data/branches.csv'

                dataset_counts[dataset_key] += 1
                dataset_sizes[dataset_key] += size

    print(f"RESUMEN DEL BUCKET: '{config.awss3.bucket}'\n" + "="*50)
    for ds, count in sorted(dataset_counts.items()):
        size_mb = round(dataset_sizes[ds] / (1024 * 1024), 2)
        if ds.endswith('/'):
            print(f"📁 Carpetas/Dataset: {ds:<35} | Archivos: {count:<6} | Tamaño: {size_mb} MB")
        else:
            print(f"📄 Archivo suelto:   {ds:<35} | Archivos: {count:<6} | Tamaño: {size_mb} MB")

    total_mb = round(total_size_bytes / (1024 * 1024), 2)
    print("="*50)
    print(f"TOTAL GENERAL: {total_files} objetos | {total_mb} MB\n")

# Ejecutar
resumir_estructura_s3()

RESUMEN DEL BUCKET: 'factored-datathon-2026-s3-157725502942-us-east-2-an'
📄 Archivo suelto:   data/branches.csv                   | Archivos: 1      | Tamaño: 0.09 MB
📁 Carpetas/Dataset: data/call_center_interactions/      | Archivos: 1097   | Tamaño: 133.26 MB
📁 Carpetas/Dataset: data/call_transcripts/              | Archivos: 1097   | Tamaño: 130.88 MB
📁 Carpetas/Dataset: data/campaign_sends/                | Archivos: 1083   | Tamaño: 310.88 MB
📁 Carpetas/Dataset: data/complaints/                    | Archivos: 1097   | Tamaño: 17.16 MB
📄 Archivo suelto:   data/customers.csv                  | Archivos: 1      | Tamaño: 44.72 MB
📄 Archivo suelto:   data/daily_exchange_rates.csv       | Archivos: 1      | Tamaño: 0.74 MB
📁 Carpetas/Dataset: data/digital_events/                | Archivos: 1097   | Tamaño: 3583.29 MB
📄 Archivo suelto:   data/marketing_campaigns.csv        | Archivos: 1      | Tamaño: 0.03 MB
📄 Archivo suelto:   data/products.csv                   | Archivos: 1      | T